# Platform Decision Matrix — How a Staff DE Chooses Technology

## Kafka vs Kinesis
Throughput | Ordering | Replay | Ops | On-Prem

In [1]:

from confluent_kafka import Producer, Consumer
import time, json

topic="citi.decision.kafka"
p=Producer({"bootstrap.servers":"localhost:9092"})
start=time.time()

for i in range(50):
    p.produce(topic,json.dumps({"id":i}).encode())
p.flush()

c=Consumer({"bootstrap.servers":"localhost:9092","group.id":"g1","auto.offset.reset":"earliest"})
c.subscribe([topic])

cnt=0
while cnt<50:
    msg=c.poll(1.0)
    if msg:
        cnt+=1
end=time.time()

print(f"Kafka roundtrip: {(end-start)*1000:.2f} ms")


Kafka roundtrip: 3107.82 ms


## Spark vs Flink

In [2]:
import os, asyncio
# Local Spark — JRE 8 + winutils (avoids JDK-17 Netty and Windows NativeIO issues)
asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
os.environ['JAVA_HOME']         = 'C:/Program Files/Java/jre1.8.0_481'
os.environ['HADOOP_HOME']       = 'C:/winutils'
os.environ['PYSPARK_PYTHON']    = 'C:/py_venv/proj_educate/Scripts/python.exe'
os.environ['PYSPARK_DRIVER_PYTHON'] = 'C:/py_venv/proj_educate/Scripts/python.exe'
os.environ['PATH']              = 'C:/winutils/bin;' + os.environ.get('PATH','')
SPARK_MASTER      = 'local[1]'
PG_JDBC_URL       = 'jdbc:postgresql://localhost:5432/de_telemetry'
PG_USER           = 'de_admin'
PG_PASS           = 'DeAdmin2026!'
KAFKA_BOOTSTRAP   = 'localhost:9092'
DRIVER_CLASSPATH  = r'C:/Users/shareuser/.ivy2/jars/org.postgresql_postgresql-42.7.4.jar;C:/Users/shareuser/.ivy2/jars/org.apache.spark_spark-sql-kafka-0-10_2.12-3.5.4.jar;C:/Users/shareuser/.ivy2/jars/org.apache.spark_spark-token-provider-kafka-0-10_2.12-3.5.4.jar;C:/Users/shareuser/.ivy2/jars/org.apache.kafka_kafka-clients-3.4.1.jar;C:/Users/shareuser/.ivy2/jars/org.lz4_lz4-java-1.8.0.jar;C:/Users/shareuser/.ivy2/jars/org.xerial.snappy_snappy-java-1.1.10.5.jar;C:/Users/shareuser/.ivy2/jars/org.apache.commons_commons-pool2-2.11.1.jar'
print('JAVA_HOME:', os.environ['JAVA_HOME'])
print('HADOOP_HOME:', os.environ['HADOOP_HOME'])


JAVA_HOME: C:/Program Files/Java/jre1.8.0_481
HADOOP_HOME: C:/winutils


## Airflow vs Prefect vs Dagster

In [3]:
# Airflow health check via REST API
import requests
try:
    resp = requests.get('http://localhost:8082/api/v1/dags', auth=('airflow', 'airflow'), timeout=10)
    if resp.status_code == 200:
        dags = resp.json().get('dags', [])
        print(f'Airflow reachable — {len(dags)} DAG(s) registered')
    else:
        print(f'Airflow returned {resp.status_code}')
except requests.exceptions.ConnectionError:
    print('Airflow not reachable — stack not running, check docker compose')


Airflow not reachable — stack not running, check docker compose


## dbt vs SQLMesh

In [4]:

import subprocess
res=subprocess.run(["C:/py_venv/proj_educate/Scripts/dbt.exe","ls"],capture_output=True,text=True)
print(res.stdout[:500])


00:40:48  Running with dbt=1.11.7
00:40:48  Encountered an error:
Runtime Error
  No dbt_project.yml found at expected path D:\Workspace\Technologies\dbt_project.yml
  Verify that each entry within packages.yml (and their transitive dependencies) contains a file named dbt_project.yml
  



## Delta vs Iceberg vs Hudi

Delta for Databricks, Iceberg for multi-engine

## Cloud Decision

AWS vs GCP vs Azure — multi-cloud by BU

## Decision Framework
if latency <1s → streaming else batch